# 第3章 注意力机制——让模型学会"读懂"上下文

> Ch02 走完了"文本 → token ID → embedding → 位置编码 → 输入张量"整条链路，数据准备好了。
> 
> 本章回答的核心问题：**一堆 token 向量摆在那里，模型怎么知道哪些 token 之间有语义关联？**


开门见山：注意力机制就是大模型核心中的核心。

如果用一句话概括大模型在干什么，就是"分析一句话里每个 token 之间的关系，然后预测下一个 token"。

注意力机制负责前半句——**分析 token 之间的关系**。没有它，模型看到的就是一堆互不相干的词，根本搞不清"it"指的是"cat"还是"mat"。

这一章我们就来拆解这个机制到底是怎么算的。

> 大模型的训练目标很简单：
> 
> 给定上文，猜下一个 token。比如看到 "I love"，应该猜出 "you"。模型内部靠什么猜？
> 
> 靠的是理解 "I"、"love" 这些 token 之间的语法和语义关系——这恰恰是注意力机制要提供的。
> 
> 所以，理解注意力，就是理解大模型怎么"看懂"一句话。

所以，我们先讨论一句话中，每个token之间的关系如何表示

# Part 1 不同token之间关系的表示

先导入需要的库，我们先用简单的例子看看

In [1]:
from importlib.metadata import version
print("torch version:", version("torch"))

torch version: 2.12.1


从上一章，我们得到了带有位置编码的输入 token 向量。举个例子，假设 embedding 维度为 3，context_length=6：

输入是一句话 "Your journey starts with one step"，已经走完 Ch02 的 embedding 和位置编码流程，得到了 6 个 token、每个 3 维的向量矩阵。

> 这里的维度故意设得很小（3 维），是为了每一步都能把完整的矩阵打印出来。真实场景（256~1600 维）数值太多，不太方便看计算过程。

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89],   # Your
   [0.55, 0.87, 0.66],   # journey
   [0.57, 0.85, 0.64],   # starts
   [0.22, 0.58, 0.33],   # with
   [0.05, 0.80, 0.55],   # one
   [0.77, 0.25, 0.10]]   # step
)

print("输入形状:", inputs.shape)
print("（6 个 token，每个 3 维）\n")
print("输入矩阵:")
print(inputs)

输入形状: torch.Size([6, 3])
（6 个 token，每个 3 维）

输入矩阵:
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.0500, 0.8000, 0.5500],
        [0.7700, 0.2500, 0.1000]])


我们如何知道 Your 和 journey 这两个 token 的关系如何呢？

它们现在都用向量表示，而向量之间的关系可以通过**点积**（dot product）来计算出一个值：

$$\alpha \cdot \beta = |\alpha||\beta| \cos \theta$$

直观理解：两个向量方向越一致（夹角越小），点积越大；方向越正交，点积越接近 0。所以我们可以用点积表示两个 token 向量的"相关程度"。

> 严格来说，点积同时受方向和向量长度影响——很长的向量即使方向不太一致，点积也可能不小。后面会通过归一化手段来缓解这个问题。这里先按"方向相似度"来理解。

In [3]:
# 计算Your 与 journey的点积
Your = inputs[0]
journey = inputs[1]

print("Your的矩阵表示: ", Your)
print("journey的矩阵表示: ", journey)

# 计算点积：
Your_journey_dot = torch.dot(Your, journey)
print("Your 与 journey的点积结果: ", Your_journey_dot)

# 手算验证一下
hand_result = 0
for i in range(3):
    hand_result = hand_result + Your[i] * journey[i]
print(
    f"torch.dot结果 = {Your_journey_dot},\n"
    f"手算结果 = {hand_result},\n"
    f"{'与手算结果一致' if Your_journey_dot == hand_result else '与手算结果不一致'}"
)

Your的矩阵表示:  tensor([0.4300, 0.1500, 0.8900])
journey的矩阵表示:  tensor([0.5500, 0.8700, 0.6600])
Your 与 journey的点积结果:  tensor(0.9544)
torch.dot结果 = 0.9544000625610352,
手算结果 = 0.9544000625610352,
与手算结果一致


既然如此，我们可以这样计算所有token之间的点积

In [4]:
dot = inputs @ inputs.T
print(dot)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.6310, 0.4576],
        [0.9544, 1.4950, 1.4754, 0.8434, 1.0865, 0.7070],
        [0.9422, 1.4754, 1.4570, 0.8296, 1.0605, 0.7154],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.6565, 0.3474],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.9450, 0.2935],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.2935, 0.6654]])


这样，只需要取Your序号行与 journey序号列，即[0]行[1]列，即可得到这两个token的向量点积结果，我们可以看到dot矩阵的[0]行[1]列内容与我们刚刚手动计算的结果一致。

点积结果就是所谓的**注意力得分**（attention scores）。不过这些得分范围很大、不好直观比较，所以我们把它归一化到 0~1 区间，让每行加起来等于 1——这样就变成了一目了然的"关注度比例"。

最简单的方式：把每个得分除以整行的总和。比如对于 Your（第 0 行），把它跟所有 token 的点积得分分别除以这一行的总分：

In [5]:
# dot[0] = Your 与其他所有 token 的点积结果（6 个值）
# 除以它们的总和 → 归一化，让结果加起来等于 1
norm_avg = dot[0] / torch.sum(dot[0])
print("Your 对各 token 的归一化关注度:")
print(norm_avg)
print(f"验证：加起来 = {norm_avg.sum():.4f}")

Your 对各 token 的归一化关注度:
tensor([0.2241, 0.2140, 0.2113, 0.1066, 0.1415, 0.1026])
验证：加起来 = 1.0000


直接除总和虽然简单，但实际项目中都用 **softmax**。原因有两个：

1. **放大差异**：softmax 的指数运算（`exp`）会让原本接近的值被拉得更开，模型更容易"聚焦"到真正相关的 token，而不是一碗水端平。
2. **梯度友好**：softmax 处处可导，配合交叉熵损失函数时梯度计算特别简洁，训练稳定。

下面先手写一个 naive 版 softmax 感受一下原理，再换成 PyTorch 的优化版。

In [6]:
# softmax 简单实现 —— 仅用于理解原理
# 注意：这个版本只支持 1D 张量（比如 dot[0] 这种一行数据）
# PyTorch 的 torch.softmax 支持任意维度，后面会用它
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)  # dim=0 对 1D 张量没问题

In [7]:
norm_softmax = softmax_naive(dot[0])
print(norm_softmax)

tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1452, 0.1220])


In [8]:
# torch.softmax 对 2D 矩阵按行归一化
# dim=1 表示沿着列方向做 softmax —— 每一行独立计算，行内所有值加起来等于 1
# dot 的形状是 [6, 6]，6 个 token 各自对 6 个 token 的关注
attn_weighs = torch.softmax(dot, dim=1)
print("softmax 归一化后的注意力权重（每行加起来 = 1）：")
print(attn_weighs)
print(f"\n验证每行和: {attn_weighs.sum(dim=1).tolist()}")

softmax 归一化后的注意力权重（每行加起来 = 1）：
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1452, 0.1220],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1581, 0.1082],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1565, 0.1108],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1720, 0.1263],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.1896, 0.0988],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1295, 0.1879]])

验证每行和: [0.9999998807907104, 1.0, 1.0000001192092896, 1.0, 1.0, 0.9999999403953552]


In [9]:
attn_weighs = torch.softmax(dot, dim=1) # dim=1：按行归一化，本身dot矩阵的维度是[batch, 行, 列]
print(attn_weighs)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1452, 0.1220],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1581, 0.1082],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1565, 0.1108],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1720, 0.1263],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.1896, 0.0988],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1295, 0.1879]])


## 有了注意力权重，下一步就是**按权重把所有 token 的信息汇总**

——对每个 token，用它对其他 token 的注意力权重，对原始向量做加权求和，得到"带上下文的新表示"。

```text
token 0 的新向量 = attn[0][0] × inputs[0]
                 + attn[0][1] × inputs[1]
                 + attn[0][2] × inputs[2]
                 + ...
```

矩阵形式就是一句话：

In [10]:
# 加权求和：注意力权重 @ 原始向量 → 带上下文的 token 表示
# attn_weighs: [6, 6] —— 每行是 token i 对 6 个 token 的注意力权重
# inputs:     [6, 3] —— 6 个 token，每个 3 维
# context:    [6, 3] —— 每个 token 得到了融合全句信息后的新向量
context = attn_weighs @ inputs

print("=== 加权求和：每个 token 的新向量 ===")
print(f"形状: {context.shape} （和原始 inputs 形状完全一样）\n")
for i in range(len(context)):
    print(f"token {i} 的新向量: {context[i].tolist()}")

print("\n对比：")
print(f"  inputs[0]（原始）: {inputs[0].tolist()}")
print(f"  context[0]（融合后）: {context[0].tolist()}")
print("  → 向量变了！它不再是 'Your' 独自的信息，而是融合了全句所有 token 的信息")

=== 加权求和：每个 token 的新向量 ===
形状: torch.Size([6, 3]) （和原始 inputs 形状完全一样）

token 0 的新向量: [0.44205939769744873, 0.593098521232605, 0.5789890885353088]
token 1 的新向量: [0.4418657422065735, 0.6514819860458374, 0.5683088898658752]
token 2 的新向量: [0.4431275427341461, 0.6495946049690247, 0.5670731067657471]
token 3 的新向量: [0.43038973212242126, 0.6298280954360962, 0.5510269999504089]
token 4 的新向量: [0.41772449016571045, 0.6503232717514038, 0.5645352602005005]
token 5 的新向量: [0.46710169315338135, 0.5909926891326904, 0.5265965461730957]

对比：
  inputs[0]（原始）: [0.4300000071525574, 0.15000000596046448, 0.8899999856948853]
  context[0]（融合后）: [0.44205939769744873, 0.593098521232605, 0.5789890885353088]
  → 向量变了！它不再是 'Your' 独自的信息，而是融合了全句所有 token 的信息


---

## Part 1 小结：我们做了什么、还差什么

到这里，你已经看到了注意力机制的**核心计算链**：

```text
原始向量 → 点积(算相似度) → softmax(归一化成权重) → 加权求和(融合上下文)
   [6,3]       [6,6]                [6,6]                 [6,3]
```

每个 token 的向量不再是孤立的——它融合了整句话里所有 token 的信息，融合比例由注意力权重决定。

但这个方案有一个根本缺陷：**注意力权重完全由输入的原始 embedding 决定，没有任何可训练的参数。

** 模型无法学会"当我是动词时该多看主语""当我是代词时该找前面的名词"——它的注意力模式被初始 embedding 锁死了。

下一节（Part 2）引入 **Q/K/V 三个可训练矩阵**，把"去问""被问""传信息"这三件事拆开，让模型自己学会怎么分配注意力。

这也就是 Transformer 论文里真正的 **scaled dot-product self-attention**。

# Part 2 如何可训练

Part 1 的三步法跑通了——`softmax(x @ x.T)` 算出权重，再用权重对 `x` 做加权求和。但它有一个致命缺陷：**整个计算链没有一个可训练参数。** 注意力模式完全由初始 embedding 的随机值决定，注意力权重完全由原始输入向量之间的点积决定，模型无法主动学习"我应该怎么看其他 token"。

就像是：现在模型看到动词时，会随机瞎看，而无法学会"当我是动词时该多看主语""当我是代词时该找前面的名词"。



而且，同一个向量 `x` 同时承担了三件事：
- 它既要作为 **Query**，去和别的 token 比相似度；
- 又要作为 **Key**，被别的 token 拿来匹配；
- 还要作为 **Value**，提供最后被汇总的信息。

也就是说，一个 token 向量既要负责"我想找什么"，又要负责"别人怎么找到我"，还要负责"我把什么内容传出去"。三个目标放在同一个向量里，表达能力就很受限。

用图书馆查资料来类比：你去图书馆查资料时，至少有三个角色——

| 角色 | 类比 | 在 Attention 里的含义 |
|------|------|---------------------|
| **Query** | 你输入的关键词 | 当前 token 想找什么信息 |
| **Key** | 书架或书籍的标签 | 每个 token 对外展示什么"标签" |
| **Value** | 书里面真正的内容 | 每个 token 实际传递什么内容 |

关键词不会直接和书的正文逐字比较，而是先和标签做匹配。匹配度越高，对应书籍的内容就越值得被取出来。

所以，**Transformer**（一个2017年的划时代意义的论文所提出的模型结构） 不再直接用原始输入 `x` 去计算注意力，而是**先通过三个可训练矩阵** `W_q`、`W_k`、`W_v`，把 `x` 投影成三个独立的向量：

> 一定要注意，这里的 `W_q`、`W_k`、`W_v` 是可训练的，也就是在训练过程，矩阵中的值是不断变化的，使模型输出更贴合我们想要的结果

```python
Q = x @ W_q    # Query:  "我想找什么"
K = x @ W_k    # Key:    "我是什么标签"
V = x @ W_v    # Value:  "我有什么内容"
```

设 `x` 形状为 `[num_tokens, emb_dim]`，`W_q`、`W_k`、`W_v` 形状均为 `[emb_dim, d_out]`，则投影后的 Q、K、V 形状都是 `[num_tokens, d_out]`。

> `d_out` 可以和 `emb_dim` 一样大（GPT 里一般就是一样的），也可以不同。这里为了展示维度可变，故意设了不一样的 `d_out`。

接下来的计算和 Part 1 几乎一模一样，只多了两步——用 Q/K 代替 x，加上缩放：

```python
# 使用QKV矩阵的注意力计算过程
attn_scores = Q @ K.T / sqrt(d_k)   # d_k = d_out，缩放防梯度消失
attn_weights = torch.softmax(attn_scores, dim=-1)
contexts = attn_weights @ V
```


```python
# part 1 注意力计算过程
attn_scores = x @ x.T # 输入矩阵与自己内积，计算注意力得分
attn_weights = torch.softmax(attn_scores, dim=-1)
contexts = attn_weights @ x
```

**对比：**

| | Part 1（无参数） | Part 2（Q/K/V） |
|---|---|---|
| 参与点积的向量 | `x` @ `x.T` | `Q` @ `K.T` |
| 被汇总的向量 | `x` | `V` |
| 是否可训练 | 否 | `W_q`, `W_k`, `W_v` 皆可训练 |
| 缩放 | 无 | 除以 `√d_k` |

这样一来，模型就可以通过训练自己学会：

如动词该多关注哪些名词，代词该找哪个先行词，哪些 token 的信号应该放大、哪些该抑制。

Q/K/V 把"找什么""怎么被找""传什么"拆成三个专职矩阵，各自沿着自己的方向优化，不再互相掣肘。

我们再用示例代码看看这个过程：

In [ ]:
# 接上文：还是那 6 个 token、3 维的 inputs
# 现在用 d_out=2（故意和 emb_dim=3 不同，展示维度可变）

import torch.nn as nn

print("传到注意力机制的矩阵：\n", inputs)
print(f"也就是 {inputs.shape[0]} 个token, 每个token的维度是 {inputs.shape[1]}")

torch.manual_seed(123)
d_in = inputs.shape[1]   # embedding维度：3
d_out = 2                # 故意设小，方便看投影效果

# 三个可训练矩阵（用 nn.Linear 而非手动 rand，因为它有更好的初始化策略）
W_query = nn.Linear(d_in, d_out, bias=False) # [3, 2]的矩阵
W_key   = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

# 投影 → Q, K, V
Q = W_query(inputs)   # [6, 3] @ [3, 2]^T → [6, 2]
K = W_key(inputs)
V = W_value(inputs)

print(f"inputs 形状: {inputs.shape}")
print(f"Q 形状: {Q.shape}")
print(f"K 形状: {K.shape}")
print(f"V 形状: {V.shape}")
print("\n每个 token 从 3 维投影到了 2 维，三个矩阵各产出不同的投影")
print(f"\nQuery:\n{Q}")
print(f"\nKey:\n{K}")
print(f"\nValue:\n{V}")

传到注意力机制的矩阵：
 tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.0500, 0.8000, 0.5500],
        [0.7700, 0.2500, 0.1000]])
也就是 6 个token, 每个token的维度是 3
inputs 形状: torch.Size([6, 3])
Q 形状: torch.Size([6, 2])
K 形状: torch.Size([6, 2])
V 形状: torch.Size([6, 2])

每个 token 从 3 维投影到了 2 维，三个矩阵各产出不同的投影

Query:
tensor([[-0.3536,  0.3965],
        [-0.3021, -0.0289],
        [-0.3015, -0.0232],
        [-0.1353, -0.0978],
        [-0.1542, -0.1499],
        [-0.2052,  0.0870]], grad_fn=<MmBackward0>)

Key:
tensor([[-0.5740,  0.2727],
        [-0.8709,  0.1008],
        [-0.8628,  0.1060],
        [-0.4789,  0.0051],
        [-0.5888, -0.0388],
        [-0.4744,  0.1696]], grad_fn=<MmBackward0>)

Value:
tensor([[-0.4519,  0.2216],
        [-0.7142, -0.1961],
        [-0.7127, -0.1971],
        [-0.3809, -0.1557],
        [-0.4213, -0.1501],
        [-0.4861, -0.1597]], grad_fn=<MmBackward0>)


In [18]:
# 走完 Q/K/V 注意力的完整计算链，和 Part 1 对比

d_k = d_out  # 缩放因子

# Step 1: 亲密度得分（Q @ K.T，再缩放）
attn_scores_v2 = Q @ K.T / (d_k ** 0.5)
print("=== Step 1: 缩放后的亲密度得分 (Q@K.T / √d_k) ===")
print(f"形状: {attn_scores_v2.shape}")
print(attn_scores_v2)

# Step 2: softmax → 注意力权重
attn_weights_v2 = torch.softmax(attn_scores_v2, dim=-1)
print("\n=== Step 2: 注意力权重 ===")
print(attn_weights_v2)
print(f"每行和: {attn_weights_v2.sum(dim=-1).tolist()}")

# Step 3: 加权求和 → 上下文向量
context_v2 = attn_weights_v2 @ V
print("\n=== Step 3: 上下文向量 (weights @ V) ===")
print(f"形状: {context_v2.shape}（从 Part 1 的 [6,3] 变成了 [6,{d_out}]，因为 V 是 {d_out} 维）")
print(context_v2)

print("\n=== Part 1 vs Part 2 对比 ===")
print(f"Part 1 context[0]: {context[0].tolist()}")
print(f"Part 2 context[0]: {context_v2[0].tolist()}")
print(f"Part 1 形状: {context.shape}  |  Part 2 形状: {context_v2.shape}")
print("\n关键区别：Part 2 的 W_q/W_k/W_v 可以被反向传播优化。训练前是随机的，")
print("训练后模型会学会'该注意谁'——这就是从 V1 到 V2 的质变。")

=== Step 1: 缩放后的亲密度得分 (Q@K.T / √d_k) ===
形状: torch.Size([6, 6])
tensor([[0.2200, 0.2460, 0.2454, 0.1212, 0.1364, 0.1661],
        [0.1170, 0.1840, 0.1822, 0.1022, 0.1266, 0.0979],
        [0.1179, 0.1840, 0.1822, 0.1020, 0.1262, 0.0983],
        [0.0361, 0.0764, 0.0752, 0.0455, 0.0590, 0.0337],
        [0.0337, 0.0843, 0.0828, 0.0517, 0.0683, 0.0337],
        [0.1000, 0.1326, 0.1317, 0.0698, 0.0830, 0.0793]],
       grad_fn=<DivBackward0>)

=== Step 2: 注意力权重 ===
tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1579, 0.1627],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1652, 0.1605],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1651, 0.1606],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1674, 0.1632],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1682, 0.1625],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1639, 0.1633]],
       grad_fn=<SoftmaxBackward0>)
每行和: [0.9999999403953552, 1.0, 0.9999999403953552, 1.0, 1.0, 1.0]

=== Step 3: 上下文向量 (weights @ V) ===
形状: torch.Size([6, 2])（从 Part 1 的 [6,3] 变成了 [6

# Part 3 — Causal Attention：不许偷看未来

## 动机：GPT 不能"抄答案"

回顾一下 GPT 的训练目标：给定前文，预测下一个 token。比如看到 `"I love"`，应该猜 `"you"`。

现在想一个问题：如果模型在预测 token 3 的时候，能直接看到 token 3 本身（也就是标准答案），它还学什么？直接抄就行了。

所以在训练时，必须强制一条规则：**预测第 i 个 token 时，只能看第 0 到第 i-1 个 token，后面的全部遮住。** 这就是 causal attention（因果注意力），也叫 masked self-attention。

## 怎么做：上三角填 `-inf`

Part 2 算出来的 `attn_scores` 是一个 `[6, 6]` 矩阵，第 i 行第 j 列 = token i 对 token j 的原始得分。我们要让第 i 行、j > i 的所有位置失效。

做法简单到只有三步：

1. 生成一个上三角 mask（对角线上方 = 1，其余 = 0）
2. 把 mask == 1 的位置的 `attn_scores` 填成 `-inf`
3. 正常做 softmax——`exp(-inf) = 0`，那些位置的权重自动变成 0，且每行剩下的权重自动重新归一化

> **mask 必须在 softmax 之前加。** 如果先 softmax 再把权重置零，行和就不再是 1 了，概率语义就坏了。

In [ ]:
# Step 1: 生成上三角 mask
# torch.triu (upper triangle): 保留上三角，diagonal=1 表示对角线上方第一行开始
# 这样对角线及下方 = 0（允许看），对角线上方 = 1（遮住）

seq_len = inputs.shape[0]
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

print("=== Causal Mask（1 = 不许看，0 = 允许看）===")
print(mask)

# Step 2: 把 mask == 1 的位置填成 -inf
attn_scores_masked = attn_scores_v2.masked_fill(mask.bool(), -float('inf'))

print("\n=== 填充 -inf 后的注意力得分 ===")
print(attn_scores_masked)
# 注意看每行的上三角全变成了 -inf

In [ ]:
# Step 3: 在 mask 之后做 softmax，然后加权求和
attn_weights_masked = torch.softmax(attn_scores_masked, dim=-1)

print("=== Causal Attention 权重（softmax 之后）===")
print(attn_weights_masked)
print(f"\n每行和: {attn_weights_masked.sum(dim=-1).tolist()}  ← 仍然是 1")

# 加权求和 → 上下文向量
context_masked = attn_weights_masked @ V

print(f"\n=== 对比：无 mask vs 有 mask 的注意力权重（第 4 行）===")
print(f"无 mask: {attn_weights_v2[4].tolist()}")
print(f"有 mask: {attn_weights_masked[4].tolist()}")
print("\n无 mask：token 4 把注意力分散到了 6 个 token（包括未来的 token 5）")
print("有 mask：token 4 只能看到 token 0~4，token 5 的权重 = 0")